# SMM4H-HeaRD 2026 – Task 2, Subtask 2: Insomnia Detection in Clinical Notes
## Multi-label Classification + Evidence Span Extraction using Qwen3-8B

**Strategy:** Zero/few-shot prompting of Qwen3-8B-Instruct with structured reasoning.
- For each clinical note, ask the model to evaluate all 4 rule components simultaneously.
- Extract character-level spans using exact text search back into the original note.
- Post-process to enforce the constraint: yes → non-empty spans, no → empty spans.

**Rule Definitions (from task resources):**
- **Definition 1:** Patient has direct insomnia symptom (can't sleep, trouble sleeping, insomnia, etc.)
- **Definition 2:** Patient has indirect/daytime impairment due to sleep issues (fatigue, restlessness due to sleep)
- **Rule B:** Patient is prescribed a primary sleep medication (Zolpidem/Ambien, Triazolam/Halcion, Temazepam, etc.)
- **Rule C:** Patient is prescribed a secondary/off-label sleep medication (Benzodiazepines like Lorazepam/Ativan, Diazepam, Quetiapine/Seroquel, Trazodone, Diphenhydramine, etc.)


In [5]:
# ─── Install dependencies ───────────────────────────────────────────────────
#!pip install -q transformers==4.47.0 accelerate==0.34.0 bitsandbytes==0.44.1 \
#              sentencepiece protobuf
!pip install -U -q transformers --upgrade accelerate bitsandbytes sentencepiece protobuf
print('Dependencies installed.')

Dependencies installed.


In [6]:
# ─── Imports ────────────────────────────────────────────────────────────────
import json, re, os, time, gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
#from google.colab import files

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA L4
VRAM: 23.7 GB


In [ ]:
# ─── Upload data files ───────────────────────────────────────────────────────
# Upload these files from your local machine:
#   - training_corpus.csv
#   - validation_corpus.csv
#   - test_corpus.csv
#   - subtask_2_train.json
#   - subtask_2_valid.json
print('Please upload your data files:')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [7]:
# ─── Load all data ────────────────────────────────────────────────────────────
train_corpus = pd.read_csv('training_corpus.csv').set_index('note_id')
valid_corpus = pd.read_csv('validation_corpus.csv').set_index('note_id')
test_corpus  = pd.read_csv('test_corpus.csv').set_index('note_id')

with open('subtask_2_train.json') as f:
    train_labels = json.load(f)

with open('subtask_2_valid.json') as f:
    valid_labels = json.load(f)

COMPONENTS = ['Definition 1', 'Definition 2', 'Rule B', 'Rule C']

print(f'Train notes: {len(train_corpus)} | Train labels: {len(train_labels)}')
print(f'Valid notes: {len(valid_corpus)} | Valid labels: {len(valid_labels)}')
print(f'Test notes:  {len(test_corpus)}')

# Label distribution in training
print('\nTraining label distribution:')
for comp in COMPONENTS:
    yes = sum(1 for v in train_labels.values() if v[comp]['label'] == 'yes')
    print(f'  {comp}: yes={yes}, no={len(train_labels)-yes}')

Train notes: 156 | Train labels: 156
Valid notes: 23 | Valid labels: 23
Test notes:  1959

Training label distribution:
  Definition 1: yes=73, no=83
  Definition 2: yes=50, no=106
  Rule B: yes=37, no=119
  Rule C: yes=67, no=89


In [8]:
# ─── Load Qwen3-8B with 4-bit quantization (fits in 16GB VRAM) ──────

MODEL_NAME = 'Qwen/Qwen3-8B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('Loading tokenizer...')
# Ensure trust_remote_code=True is here
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print('Loading model (4-bit quantized)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True, # This is the critical line
)
model.eval()
print('Model loaded!')
print(f'Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading tokenizer...


Loading model (4-bit quantized)...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model loaded!
Memory used: 6.09 GB


In [9]:
# ─── Prompt template & inference ─────────────────────────────────────────────

SYSTEM_PROMPT = """You are a clinical NLP expert specializing in identifying insomnia in clinical notes.

You will analyze a clinical note and evaluate it against 4 rule components. Return ONLY valid JSON.

RULE DEFINITIONS:
- Definition 1: The patient has DIRECT insomnia symptoms mentioned. Look for: insomnia, unable to sleep, can't sleep, difficulty sleeping, not sleeping, sleeplessness, poor sleep, sleep disturbance, sleep problems, trouble sleeping, waking up at night, difficulty falling asleep, difficulty staying asleep.
- Definition 2: The patient has INDIRECT sleep problems / daytime impairment from sleep issues. Look for: fatigue due to sleep, restlessness (when sleep-related), exhaustion, daytime sleepiness, upset/agitated causing inability to sleep, sleep deprivation effects.
- Rule B: The patient is prescribed a PRIMARY sleep/hypnotic medication. Includes: Zolpidem (Ambien, Edluar, Intermezzo), Triazolam (Halcion), Temazepam (Restoril), Eszopiclone (Lunesta), Zaleplon (Sonata), Flurazepam (Dalmane), Estazolam, Quazepam, Ramelteon (Rozerem).
- Rule C: The patient is prescribed a SECONDARY (off-label) sleep medication. Includes: Lorazepam (Ativan), Diazepam (Valium), Clonazepam (Klonopin), Alprazolam (Xanax), Midazolam (Versed), Quetiapine (Seroquel/Seraquil), Trazodone (traZODONE), Diphenhydramine (Benadryl), Mirtazapine (Remeron), Doxepin, Melatonin, Hydroxyzine (Vistaril).

IMPORTANT RULES:
1. Only label 'yes' if you find CLEAR evidence in the text.
2. For each 'yes', provide the EXACT text snippets (verbatim) that support your decision.
3. For Rule B and Rule C, find the medication names as they appear in the text.
4. A medication in the header prescription list counts as evidence for Rule B/C.
5. If label is 'no', evidence_texts must be empty.

Output format (JSON only, no other text):
{
  "Definition 1": {"label": "yes" or "no", "evidence_texts": ["exact text from note", ...]},
  "Definition 2": {"label": "yes" or "no", "evidence_texts": ["exact text from note", ...]},
  "Rule B": {"label": "yes" or "no", "evidence_texts": ["exact text from note", ...]},
  "Rule C": {"label": "yes" or "no", "evidence_texts": ["exact text from note", ...]}
}"""


def build_user_prompt(text):
    # Truncate very long notes to avoid OOM (keep first 12000 chars)
    truncated = text[:12000]
    return f"""Analyze this clinical note and evaluate each rule component. Return ONLY JSON.

CLINICAL NOTE:
{truncated}

Evaluate Definition 1, Definition 2, Rule B, Rule C and return JSON:"""


def run_inference(text, max_new_tokens=1024, temperature=0.1):
    """Run model inference and return raw text output."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(text)}
    ]
    # Qwen3 uses /no_think token to disable chain-of-thought (faster, more structured)
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        # Disable thinking mode for structured output
        enable_thinking=False  # Qwen3 specific: skips <think> block
    )
    inputs = tokenizer(formatted, return_tensors='pt', truncation=True,
                       max_length=6000).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)


print('Prompt and inference functions defined.')

Prompt and inference functions defined.


In [10]:
# ─── Span extraction utilities ────────────────────────────────────────────────

def find_all_spans(text, evidence_text):
    """
    Find all character-level spans of evidence_text in text.
    Returns list of 'start end' strings.
    Case-insensitive search.
    """
    spans = []
    et = evidence_text.strip()
    if not et:
        return spans
    # Try exact match first
    start = 0
    lower_text = text.lower()
    lower_et = et.lower()
    while True:
        idx = lower_text.find(lower_et, start)
        if idx == -1:
            break
        spans.append(f"{idx} {idx + len(et)}")
        start = idx + 1
    return spans


def find_best_span(text, evidence_text):
    """
    Find the first occurrence of evidence_text in text.
    Falls back to fuzzy substring search if exact not found.
    """
    spans = find_all_spans(text, evidence_text)
    if spans:
        return spans  # return all occurrences

    # Fallback: try trimming and searching for a shorter version
    words = evidence_text.strip().split()
    if len(words) > 1:
        # Try first word match
        for w in words:
            if len(w) >= 4:  # skip very short words
                s = find_all_spans(text, w)
                if s:
                    return [s[0]]  # return first word match
    return []


def parse_model_output(raw_output, note_text):
    """
    Parse JSON output from model and convert evidence_texts to character spans.
    Returns a dict in the submission format.
    """
    result = {}

    # Extract JSON from output (model might include extra text)
    json_match = re.search(r'\{[\s\S]*\}', raw_output)
    if not json_match:
        # Return all-no if can't parse
        return {comp: {'label': 'no', 'span': []} for comp in COMPONENTS}

    try:
        parsed = json.loads(json_match.group(0))
    except json.JSONDecodeError:
        # Try to fix common JSON issues
        try:
            fixed = json_match.group(0).replace("'", '"')
            parsed = json.loads(fixed)
        except:
            return {comp: {'label': 'no', 'span': []} for comp in COMPONENTS}

    for comp in COMPONENTS:
        if comp not in parsed:
            result[comp] = {'label': 'no', 'span': []}
            continue

        comp_data = parsed[comp]
        label = comp_data.get('label', 'no').lower().strip()
        if label not in ('yes', 'no'):
            label = 'no'

        evidence_texts = comp_data.get('evidence_texts', [])

        if label == 'no':
            result[comp] = {'label': 'no', 'span': []}
            continue

        # Find character spans for all evidence texts
        all_spans = []
        for et in evidence_texts:
            if isinstance(et, str) and et.strip():
                spans = find_best_span(note_text, et)
                all_spans.extend(spans)

        # Deduplicate spans
        all_spans = list(dict.fromkeys(all_spans))

        if not all_spans:
            # Model said yes but we couldn't find spans - demote to no
            # This is critical: yes requires non-empty span
            label = 'no'

        result[comp] = {'label': label, 'span': all_spans}

    return result


print('Span extraction utilities defined.')

Span extraction utilities defined.


In [11]:
# ─── Evaluation metrics ────────────────────────────────────────────────────────

def compute_label_metrics(predictions, gold):
    """
    Compute micro-average Precision, Recall, F1 for label classification.
    predictions and gold are dicts: note_id -> {comp -> {label, span}}
    """
    tp = fp = fn = 0
    for note_id in gold:
        if note_id not in predictions:
            for comp in COMPONENTS:
                if gold[note_id][comp]['label'] == 'yes':
                    fn += 1
            continue
        for comp in COMPONENTS:
            g_label = gold[note_id][comp]['label']
            p_label = predictions[note_id][comp]['label']
            if g_label == 'yes' and p_label == 'yes':
                tp += 1
            elif g_label == 'no' and p_label == 'yes':
                fp += 1
            elif g_label == 'yes' and p_label == 'no':
                fn += 1

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0.0
    return {'precision': prec, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}


def spans_overlap(span1, span2):
    """Check if two 'start end' spans overlap."""
    s1, e1 = map(int, span1.split())
    s2, e2 = map(int, span2.split())
    return s1 < e2 and s2 < e1


def compute_span_metrics(predictions, gold, match='partial'):
    """
    Compute micro-average span Precision, Recall, F1.
    match='exact' for exact match, 'partial' for overlap.
    """
    tp = fp = fn = 0
    for note_id in gold:
        if note_id not in predictions:
            for comp in COMPONENTS:
                fn += len(gold[note_id][comp].get('span', []))
            continue
        for comp in COMPONENTS:
            g_spans = gold[note_id][comp].get('span', [])
            p_spans = predictions[note_id][comp].get('span', [])

            matched_gold = set()
            matched_pred = set()

            for i, gs in enumerate(g_spans):
                for j, ps in enumerate(p_spans):
                    if match == 'exact':
                        hit = (gs == ps)
                    else:
                        hit = spans_overlap(gs, ps)
                    if hit:
                        matched_gold.add(i)
                        matched_pred.add(j)

            tp += len(matched_gold)
            fp += len(p_spans) - len(matched_pred)
            fn += len(g_spans) - len(matched_gold)

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0.0
    return {'precision': prec, 'recall': rec, 'f1': f1}


def evaluate(predictions, gold, split_name='Validation'):
    label_m = compute_label_metrics(predictions, gold)
    span_exact = compute_span_metrics(predictions, gold, 'exact')
    span_partial = compute_span_metrics(predictions, gold, 'partial')
    print(f'\n=== {split_name} Results ===')
    print(f'Label   P={label_m["precision"]:.4f}  R={label_m["recall"]:.4f}  F1={label_m["f1"]:.4f}')
    print(f'Span(E) P={span_exact["precision"]:.4f}  R={span_exact["recall"]:.4f}  F1={span_exact["f1"]:.4f}')
    print(f'Span(P) P={span_partial["precision"]:.4f}  R={span_partial["recall"]:.4f}  F1={span_partial["f1"]:.4f}')
    return label_m, span_exact, span_partial


print('Evaluation functions defined.')

Evaluation functions defined.


In [12]:
# ─── Run on Validation Set ────────────────────────────────────────────────────
# Process all validation notes (23 notes)

valid_predictions = {}
errors = []

valid_ids = list(valid_labels.keys())
print(f'Running inference on {len(valid_ids)} validation notes...')

for i, note_id in enumerate(valid_ids):
    note_id_int = int(note_id)
    if note_id_int not in valid_corpus.index:
        print(f'  [{i+1}/{len(valid_ids)}] {note_id}: NOT FOUND in corpus, skipping')
        errors.append(note_id)
        continue

    text = valid_corpus.loc[note_id_int, 'text']

    try:
        raw = run_inference(text)
        result = parse_model_output(raw, text)
        valid_predictions[note_id] = result

        # Quick check
        labels_str = ' | '.join(f'{c[:4]}:{result[c]["label"]}' for c in COMPONENTS)
        print(f'  [{i+1}/{len(valid_ids)}] {note_id}: {labels_str}')
    except Exception as e:
        print(f'  [{i+1}/{len(valid_ids)}] {note_id}: ERROR - {e}')
        errors.append(note_id)
        valid_predictions[note_id] = {comp: {'label': 'no', 'span': []} for comp in COMPONENTS}

    # Clear cache periodically
    if (i + 1) % 5 == 0:
        gc.collect()
        torch.cuda.empty_cache()

print(f'\nDone. Errors: {len(errors)}')

Running inference on 23 validation notes...


  [1/23] 10686: Defi:no | Defi:no | Rule:no | Rule:no
  [2/23] 1262007: Defi:yes | Defi:no | Rule:yes | Rule:yes
  [3/23] 1272946: Defi:yes | Defi:no | Rule:no | Rule:yes
  [4/23] 1279443: Defi:yes | Defi:no | Rule:no | Rule:no
  [5/23] 1281746: Defi:no | Defi:no | Rule:no | Rule:no
  [6/23] 1284059: Defi:yes | Defi:yes | Rule:no | Rule:yes
  [7/23] 1288748: Defi:yes | Defi:no | Rule:no | Rule:yes
  [8/23] 1320608: Defi:yes | Defi:no | Rule:no | Rule:yes
  [9/23] 1324558: Defi:yes | Defi:no | Rule:no | Rule:yes
  [10/23] 1328271: Defi:no | Defi:no | Rule:yes | Rule:yes
  [11/23] 1328300: Defi:yes | Defi:no | Rule:yes | Rule:yes
  [12/23] 1333759: Defi:yes | Defi:no | Rule:no | Rule:yes
  [13/23] 1334025: Defi:yes | Defi:no | Rule:yes | Rule:yes
  [14/23] 2073: Defi:no | Defi:no | Rule:no | Rule:no
  [15/23] 24581: Defi:no | Defi:no | Rule:no | Rule:yes
  [16/23] 25898: Defi:no | Defi:no | Rule:no | Rule:yes
  [17/23] 45094: Defi:no | Defi:no | Rule:no | Rule:no
  [18/23] 54155: Defi:no

In [13]:
# ─── Evaluate on Validation ────────────────────────────────────────────────────
label_m, span_exact, span_partial = evaluate(valid_predictions, valid_labels, 'Validation')

# Per-component breakdown
print('\n--- Per-component Label Accuracy ---')
for comp in COMPONENTS:
    correct = sum(
        1 for nid in valid_labels
        if nid in valid_predictions and
           valid_predictions[nid][comp]['label'] == valid_labels[nid][comp]['label']
    )
    total = len(valid_labels)
    print(f'  {comp}: {correct}/{total} = {correct/total:.2%}')


=== Validation Results ===
Label   P=0.6000  R=0.7826  F1=0.6792
Span(E) P=0.1087  R=0.1351  F1=0.1205
Span(P) P=0.4565  R=0.5676  F1=0.5060

--- Per-component Label Accuracy ---
  Definition 1: 22/23 = 95.65%
  Definition 2: 19/23 = 82.61%
  Rule B: 19/23 = 82.61%
  Rule C: 15/23 = 65.22%


In [14]:
# ─── Error Analysis (optional) ───────────────────────────────────────────────
print('=== Validation Error Analysis ===')
for nid in valid_labels:
    if nid not in valid_predictions:
        continue
    for comp in COMPONENTS:
        g = valid_labels[nid][comp]['label']
        p = valid_predictions[nid][comp]['label']
        if g != p:
            g_text = valid_labels[nid][comp].get('text', [])
            print(f'  Note {nid} | {comp}: Gold={g}, Pred={p} | Gold text: {g_text}')

=== Validation Error Analysis ===
  Note 1262007 | Rule C: Gold=no, Pred=yes | Gold text: []
  Note 1272946 | Definition 2: Gold=yes, Pred=no | Gold text: ['upset with what has been going on with her', 'Restless']
  Note 1272946 | Rule C: Gold=no, Pred=yes | Gold text: []
  Note 1279443 | Definition 2: Gold=yes, Pred=no | Gold text: ['insomnia']
  Note 1279443 | Rule B: Gold=yes, Pred=no | Gold text: ['halcion']
  Note 1284059 | Definition 2: Gold=no, Pred=yes | Gold text: []
  Note 1288748 | Rule B: Gold=yes, Pred=no | Gold text: ['Zolpidem']
  Note 1324558 | Definition 1: Gold=no, Pred=yes | Gold text: []
  Note 1324558 | Rule C: Gold=no, Pred=yes | Gold text: []
  Note 1328271 | Rule B: Gold=no, Pred=yes | Gold text: []
  Note 1328271 | Rule C: Gold=no, Pred=yes | Gold text: []
  Note 1328300 | Definition 2: Gold=yes, Pred=no | Gold text: ['insomnia']
  Note 1328300 | Rule B: Gold=no, Pred=yes | Gold text: []
  Note 24581 | Rule C: Gold=no, Pred=yes | Gold text: []
  Note 25898 | Ru

In [15]:
# ─── Run on Full Test Set ──────────────────────────────────────────────────────
# WARNING: 1959 notes – this will take ~4-6 hours on T4 GPU.
# Save checkpoints every 50 notes to avoid losing progress.

CHECKPOINT_FILE = 'test_predictions_checkpoint.json'
BATCH_SIZE_CHECKPOINT = 50  # save every N notes

# Load checkpoint if exists
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        test_predictions = json.load(f)
    print(f'Loaded checkpoint: {len(test_predictions)} notes already processed.')
else:
    test_predictions = {}

test_ids = [str(nid) for nid in test_corpus.index.tolist()]
remaining = [nid for nid in test_ids if nid not in test_predictions]
print(f'Total test notes: {len(test_ids)} | Remaining: {len(remaining)}')

start_time = time.time()

for i, note_id in enumerate(remaining):
    note_id_int = int(note_id)
    if note_id_int not in test_corpus.index:
        test_predictions[note_id] = {comp: {'label': 'no', 'span': []} for comp in COMPONENTS}
        continue

    text = test_corpus.loc[note_id_int, 'text']

    try:
        raw = run_inference(text)
        result = parse_model_output(raw, text)
        test_predictions[note_id] = result
    except Exception as e:
        print(f'ERROR on {note_id}: {e}')
        test_predictions[note_id] = {comp: {'label': 'no', 'span': []} for comp in COMPONENTS}

    # Progress & ETA
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        eta = (len(remaining) - i - 1) / rate / 60
        print(f'  [{i+1}/{len(remaining)}] ETA: {eta:.0f} min')

    # Checkpoint
    if (i + 1) % BATCH_SIZE_CHECKPOINT == 0:
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump(test_predictions, f)
        print(f'  Checkpoint saved at {i+1} notes.')

    # Clear GPU cache
    if (i + 1) % 20 == 0:
        gc.collect()
        torch.cuda.empty_cache()

# Final save
with open(CHECKPOINT_FILE, 'w') as f:
    json.dump(test_predictions, f)

print(f'\nTest inference complete! {len(test_predictions)} notes processed.')
print(f'Total time: {(time.time()-start_time)/60:.1f} min')

Loaded checkpoint: 1959 notes already processed.
Total test notes: 1959 | Remaining: 0

Test inference complete! 1959 notes processed.
Total time: 0.0 min


In [16]:
# ─── Format Submission Output ──────────────────────────────────────────────────
# The submission format must match sample_2.json exactly:
# { note_id: { comp: { "label": "yes"/"no", "span": ["start end", ...] } } }

def format_submission(predictions):
    """Ensure submission format is correct."""
    submission = {}
    for note_id, note_preds in predictions.items():
        submission[str(note_id)] = {}
        for comp in COMPONENTS:
            comp_pred = note_preds.get(comp, {'label': 'no', 'span': []})
            label = comp_pred.get('label', 'no')
            spans = comp_pred.get('span', [])

            # Enforce constraint: yes → non-empty spans, no → empty
            if label == 'yes' and not spans:
                label = 'no'
            if label == 'no':
                spans = []

            submission[str(note_id)][comp] = {
                'label': label,
                'span': spans
            }
    return submission


submission = format_submission(test_predictions)

# Verify all test notes are in submission
missing = [str(nid) for nid in test_corpus.index if str(nid) not in submission]
print(f'Test notes: {len(test_corpus)} | In submission: {len(submission)} | Missing: {len(missing)}')

# Add missing notes as all-no
for nid in missing:
    submission[str(nid)] = {comp: {'label': 'no', 'span': []} for comp in COMPONENTS}

# Label distribution in predictions
print('\nPrediction label distribution:')
for comp in COMPONENTS:
    yes = sum(1 for v in submission.values() if v[comp]['label'] == 'yes')
    print(f'  {comp}: yes={yes} ({yes/len(submission):.1%}), no={len(submission)-yes}')

Test notes: 1959 | In submission: 1959 | Missing: 0

Prediction label distribution:
  Definition 1: yes=266 (13.6%), no=1693
  Definition 2: yes=194 (9.9%), no=1765
  Rule B: yes=363 (18.5%), no=1596
  Rule C: yes=1020 (52.1%), no=939


In [18]:
# ─── Save Submission File ──────────────────────────────────────────────────────
SUBMISSION_FILE = 'subtask_2_predictions.json'

with open(SUBMISSION_FILE, 'w') as f:
    json.dump(submission, f, indent=2)

print(f'Submission saved to {SUBMISSION_FILE}')
print(f'Total notes: {len(submission)}')

# Validate format against sample_2.json structure
sample_note_id = list(submission.keys())[0]
print(f'\nSample entry ({sample_note_id}):')
print(json.dumps(submission[sample_note_id], indent=2))

# Download
#files.download(SUBMISSION_FILE)
#print('\nDownload started!')

Submission saved to subtask_2_predictions.json
Total notes: 1959

Sample entry (20):
{
  "Definition 1": {
    "label": "no",
    "span": []
  },
  "Definition 2": {
    "label": "yes",
    "span": [
      "4675 4684",
      "5942 5964",
      "5917 5927"
    ]
  },
  "Rule B": {
    "label": "yes",
    "span": [
      "367 384",
      "6591 6619"
    ]
  },
  "Rule C": {
    "label": "no",
    "span": []
  }
}


In [19]:
# ─── Validate Submission Format ───────────────────────────────────────────────
# Run this to check your submission is correctly formatted before uploading to CodaBench

def validate_submission_format(submission, required_note_ids=None):
    errors = []

    for note_id, note_data in submission.items():
        for comp in COMPONENTS:
            if comp not in note_data:
                errors.append(f'{note_id}: missing component {comp}')
                continue

            comp_data = note_data[comp]

            if 'label' not in comp_data:
                errors.append(f'{note_id}/{comp}: missing label')
            elif comp_data['label'] not in ('yes', 'no'):
                errors.append(f'{note_id}/{comp}: invalid label {comp_data["label"]}')

            if 'span' not in comp_data:
                errors.append(f'{note_id}/{comp}: missing span')
            elif not isinstance(comp_data['span'], list):
                errors.append(f'{note_id}/{comp}: span must be a list')
            else:
                # Check span consistency with label
                if comp_data.get('label') == 'yes' and not comp_data['span']:
                    errors.append(f'{note_id}/{comp}: label=yes but span is empty')
                if comp_data.get('label') == 'no' and comp_data['span']:
                    errors.append(f'{note_id}/{comp}: label=no but span is non-empty')
                # Check span format
                for span in comp_data['span']:
                    if not re.match(r'^\d+ \d+$', str(span)):
                        errors.append(f'{note_id}/{comp}: invalid span format: {span}')

    if errors:
        print(f'VALIDATION ERRORS ({len(errors)}):')
        for e in errors[:20]:  # show first 20
            print(f'  {e}')
        if len(errors) > 20:
            print(f'  ... and {len(errors)-20} more')
    else:
        print(f'Submission format is VALID! ({len(submission)} notes, all components correct)')

    return len(errors) == 0


is_valid = validate_submission_format(submission)
print(f'\nValid: {is_valid}')

Submission format is VALID! (1959 notes, all components correct)

Valid: True


## Tips for Improving Results

### 1. Thinking Mode (slower but potentially better)
If time allows, try setting `enable_thinking=True` in `run_inference()`. Qwen3 will use its internal chain-of-thought reasoning before answering, which can improve accuracy on complex notes. This will be ~2x slower.

### 2. Few-shot Examples in Prompt
Add 2-3 examples from the training set into the system prompt to guide the model. Pick one positive and one negative example per component.

### 3. Temperature Tuning
Lower temperature (0.05–0.1) gives more deterministic outputs. Try `temperature=0.05` if you see inconsistencies.

### 4. Ensemble
Run inference 3 times per note with `temperature=0.3` and take a majority vote for labels + union of spans.

### 5. Rule-based Post-processing
For Rule B and Rule C specifically, you can add a rule-based keyword search to catch any misses:
- Rule B keywords: `zolpidem, ambien, halcion, temazepam, triazolam, eszopiclone, lunesta, zaleplon`
- Rule C keywords: `lorazepam, ativan, diazepam, valium, quetiapine, seroquel, trazodone, diphenhydramine, benadryl, midazolam, clonazepam`

### 6. Span Refinement
If the model returns a long evidence text, try to find the shortest substring that contains the key word.

### CodaBench Submission
Upload `subtask_2_predictions.json` at: https://www.codabench.org/competitions/15299


In [20]:
# ─── Optional: Rule-Based Post-processing for Rule B and Rule C ───────────────
# This uses keyword matching as a safety net to catch any model misses.
# Especially effective for Rule B and Rule C since they are drug name lookups.

RULE_B_KEYWORDS = [
    'zolpidem', 'ambien', 'intermezzo', 'edluar',
    'triazolam', 'halcion',
    'temazepam', 'restoril',
    'eszopiclone', 'lunesta',
    'zaleplon', 'sonata',
    'flurazepam', 'dalmane',
    'estazolam', 'prosom',
    'quazepam', 'doral',
    'ramelteon', 'rozerem',
    'suvorexant', 'belsomra',
    'lemborexant', 'dayvigo',
]

RULE_C_KEYWORDS = [
    'lorazepam', 'ativan',
    'diazepam', 'valium',
    'clonazepam', 'klonopin',
    'alprazolam', 'xanax',
    'midazolam', 'versed',
    'quetiapine', 'seroquel', 'seraquil',
    'trazodone', 'trazodol', 'trazODONE', 'traZODONE',
    'diphenhydramine', 'benadryl',
    'mirtazapine', 'remeron',
    'doxepin', 'silenor',
    'hydroxyzine', 'vistaril', 'atarax',
    'melatonin',
    'chloral hydrate',
]

DEF1_KEYWORDS = [
    'insomnia', 'unable to sleep', "can't sleep", 'cannot sleep',
    'difficulty sleeping', 'trouble sleeping', 'not sleeping',
    'sleeplessness', 'poor sleep', 'difficulty falling asleep',
    'difficulty staying asleep', 'sleep disturbance', 'sleep problem',
    'not slept', 'has not slept', 'unable to fall asleep',
    'not sleeping well', 'sleep poorly', 'sleep poorly',
]

DEF2_KEYWORDS = [
    'restless', 'agitated', 'upset', 'anxious', 'anxiety',
    'fatigue', 'fatigued', 'exhausted', 'daytime sleepiness',
    'sleep wake reversal', 'sleep-wake reversal',
]


def rule_based_check(text, keywords):
    """Find first keyword match in text (case-insensitive). Returns (found, spans)."""
    text_lower = text.lower()
    found_spans = []
    for kw in keywords:
        kw_lower = kw.lower()
        idx = 0
        while True:
            pos = text_lower.find(kw_lower, idx)
            if pos == -1:
                break
            found_spans.append(f"{pos} {pos + len(kw)}")
            idx = pos + 1
    return len(found_spans) > 0, found_spans


def apply_rule_based_postprocess(submission, corpus_df):
    """
    Apply rule-based post-processing to catch model misses.
    If model said 'no' but rule-based says 'yes', upgrade to 'yes'.
    If model said 'yes' but no spans found, verify with rule-based.
    """
    upgraded = 0
    new_submission = {}

    comp_keywords = {
        'Definition 1': DEF1_KEYWORDS,
        'Definition 2': DEF2_KEYWORDS,  # Only use as fallback, not primary
        'Rule B': RULE_B_KEYWORDS,
        'Rule C': RULE_C_KEYWORDS,
    }
    # Components where rule-based is highly reliable (drug names are deterministic)
    HIGH_CONFIDENCE_RULE_BASED = {'Rule B', 'Rule C', 'Definition 1'}

    for note_id, note_preds in submission.items():
        note_id_int = int(note_id)
        if note_id_int not in corpus_df.index:
            new_submission[note_id] = note_preds
            continue

        text = corpus_df.loc[note_id_int, 'text']
        new_note = {}

        for comp in COMPONENTS:
            pred = note_preds[comp]
            current_label = pred['label']
            current_spans = pred['span']

            if comp in HIGH_CONFIDENCE_RULE_BASED:
                rb_found, rb_spans = rule_based_check(text, comp_keywords[comp])

                if current_label == 'no' and rb_found:
                    # Upgrade: model missed it but rule-based found it
                    new_note[comp] = {'label': 'yes', 'span': rb_spans[:5]}  # limit spans
                    upgraded += 1
                elif current_label == 'yes' and not current_spans and rb_found:
                    # Fix: model said yes but no spans, use rule-based spans
                    new_note[comp] = {'label': 'yes', 'span': rb_spans[:5]}
                else:
                    new_note[comp] = pred
            else:
                new_note[comp] = pred

        new_submission[note_id] = new_note

    print(f'Rule-based post-processing: {upgraded} label upgrades applied.')
    return new_submission


# Apply to validation to see impact
print('=== Before Rule-Based Post-processing ===')
evaluate(valid_predictions, valid_labels, 'Validation')

valid_preds_rb = apply_rule_based_postprocess(valid_predictions, valid_corpus)
print('\n=== After Rule-Based Post-processing ===')
evaluate(valid_preds_rb, valid_labels, 'Validation')

=== Before Rule-Based Post-processing ===

=== Validation Results ===
Label   P=0.6000  R=0.7826  F1=0.6792
Span(E) P=0.1087  R=0.1351  F1=0.1205
Span(P) P=0.4565  R=0.5676  F1=0.5060
Rule-based post-processing: 3 label upgrades applied.

=== After Rule-Based Post-processing ===

=== Validation Results ===
Label   P=0.6061  R=0.8696  F1=0.7143
Span(E) P=0.1373  R=0.1892  F1=0.1591
Span(P) P=0.4510  R=0.6216  F1=0.5227


({'precision': 0.6060606060606061,
  'recall': 0.8695652173913043,
  'f1': 0.7142857142857143,
  'tp': 20,
  'fp': 13,
  'fn': 3},
 {'precision': 0.13725490196078433,
  'recall': 0.1891891891891892,
  'f1': 0.1590909090909091},
 {'precision': 0.45098039215686275,
  'recall': 0.6216216216216216,
  'f1': 0.5227272727272727})

In [22]:
# ─── Apply Rule-Based Post-processing to Test & Save Final Submission ──────────
# Only apply if it improved validation scores above!

USE_RULE_BASED = True  # Set to False if rule-based hurt validation F1

if USE_RULE_BASED:
    final_submission = apply_rule_based_postprocess(submission, test_corpus)
    print('Applied rule-based post-processing to test set.')
else:
    final_submission = submission
    print('Using model-only predictions.')

# Validate
validate_submission_format(final_submission)

# Label distribution
print('\nFinal prediction label distribution:')
for comp in COMPONENTS:
    yes = sum(1 for v in final_submission.values() if v[comp]['label'] == 'yes')
    print(f'  {comp}: yes={yes} ({yes/len(final_submission):.1%})')

# Save
FINAL_FILE = 'subtask_2_final_submission.json'
with open(FINAL_FILE, 'w') as f:
    json.dump(final_submission, f, indent=2)

#print(f'\nFinal submission saved: {FINAL_FILE}')
#files.download(FINAL_FILE)

Rule-based post-processing: 505 label upgrades applied.
Applied rule-based post-processing to test set.
Submission format is VALID! (1959 notes, all components correct)

Final prediction label distribution:
  Definition 1: yes=547 (27.9%)
  Definition 2: yes=194 (9.9%)
  Rule B: yes=413 (21.1%)
  Rule C: yes=1194 (60.9%)


In [1]:
import json

# 1. Name of your current submission file (change if it's subtask_2.json)
input_file = "subtask_2_final_submission.json" 
output_file = "fixed_subtask_2_submission.json"

with open(input_file, "r") as f:
    predictions = json.load(f)

# 2. Loop through and truncate any spans greater than 20
for note_id, rules in predictions.items():
    for rule_name, rule_data in rules.items():
        if "span" in rule_data and len(rule_data["span"]) > 20:
            print(f"Fixing Note {note_id} | {rule_name}: Truncating {len(rule_data['span'])} spans down to 20.")
            rule_data["span"] = rule_data["span"][:20] # Keep only the first 20

# 3. Save the fixed predictions
with open(output_file, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Done! Please submit '{output_file}' to CodaBench.")

Fixing Note 5884 | Rule B: Truncating 22 spans down to 20.
Fixing Note 5884 | Rule C: Truncating 25 spans down to 20.
Fixing Note 15672 | Definition 2: Truncating 29 spans down to 20.
Fixing Note 16271 | Rule C: Truncating 21 spans down to 20.
Fixing Note 30928 | Rule C: Truncating 36 spans down to 20.
Fixing Note 38828 | Rule C: Truncating 45 spans down to 20.
Fixing Note 38991 | Rule C: Truncating 27 spans down to 20.
Fixing Note 41075 | Rule C: Truncating 44 spans down to 20.
Fixing Note 1316076 | Rule C: Truncating 21 spans down to 20.
Fixing Note 1371935 | Rule C: Truncating 33 spans down to 20.
Fixing Note 1468166 | Rule B: Truncating 28 spans down to 20.
Fixing Note 1489164 | Rule C: Truncating 48 spans down to 20.
Fixing Note 1489277 | Rule C: Truncating 24 spans down to 20.
Fixing Note 1502851 | Rule C: Truncating 35 spans down to 20.
Fixing Note 1517594 | Rule C: Truncating 22 spans down to 20.
Fixing Note 1550268 | Rule C: Truncating 25 spans down to 20.
Fixing Note 1578352 